In [5]:
# Load dataset
import pandas as pd
df_raw_final = pd.read_csv(r'D:\Study\KLTN\G8Vitamin\data\raw\raw_final.csv')

In [6]:
# Check information dataset
df_raw_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30065 entries, 0 to 30064
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   SEQN                      30065 non-null  float64
 1   Gender                    30065 non-null  float64
 2   Age                       30065 non-null  float64
 3   Race                      30065 non-null  float64
 4   PIR                       27524 non-null  float64
 5   Weight                    29603 non-null  float64
 6   Height                    29635 non-null  float64
 7   BMI                       29511 non-null  float64
 8   WaistCircumference        28600 non-null  float64
 9   Hba1c                     28397 non-null  float64
 10  FastingGlucose            28302 non-null  float64
 11  Albumin                   27953 non-null  float64
 12  ALT                       27923 non-null  float64
 13  AST                       27912 non-null  float64
 14  Alkali

In [7]:
threshold = 1e-10

# Select numeric columns
numeric_cols = df_raw_final.select_dtypes(include='number').columns

# Apply threshold replacement only on numeric columns
df_raw_final[numeric_cols] = df_raw_final[numeric_cols].applymap(
    lambda x: 0 if abs(x) < threshold else x
)

C:\Users\duyp6\AppData\Local\Temp\ipykernel_21640\146079513.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_raw_final[numeric_cols] = df_raw_final[numeric_cols].applymap(


In [8]:
# === Step 1: Extract string to get first year in cycle "2001-2002" 
df_raw_final['YearStart'] = df_raw_final['YearID'].str[:4].astype(int)
df_raw_final = df_raw_final.drop(columns=['YearID'])

#  === Step 2: Create label from VitaminD ===
def binary_vitamin_d_label(row):
    val = row['VitaminD']
    year = row['YearStart']

    if pd.isna(val) or pd.isna(year):
        return None  

    return 1 if val < 50 else 0

df_raw_final['label'] = df_raw_final.apply(binary_vitamin_d_label, axis=1)

# === Step 3: Get data from 2013 previous ===
df_raw_final_train = df_raw_final[df_raw_final['YearStart'] <= 2013]
df_raw_final_test = df_raw_final[df_raw_final['YearStart'] > 2013]

#  Drop null row if it contains any NaN value
df_raw_final_train = df_raw_final_train.dropna()
df_raw_final_test = df_raw_final_test.dropna()

# === Step 4: Write to CSV file ===
df_raw_final_train.to_csv(r'D:\Study\KLTN\G8Vitamin\data\raw\raw_final_train.csv', index=False)
df_raw_final_test.to_csv(r'D:\Study\KLTN\G8Vitamin\data\raw\raw_final_test.csv', index=False)

# === Log information line number ===
print(f"✅ Số dòng train: {len(df_raw_final_train)} được lưu vào raw_final_train.csv")
print(f"✅ Số dòng test : {len(df_raw_final_test)} được lưu vào raw_final_test.csv")

✅ Số dòng train: 18669 được lưu vào raw_final_train.csv
✅ Số dòng test : 4644 được lưu vào raw_final_test.csv
